In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumRegister, ClassicalRegister, QuantumCircuit, Parameter
from qiskit.result import marginal_counts
from qiskit.visualization import plot_histogram
import numpy as np
from qiskit_aer.noise import NoiseModel

from scipy.stats import linregress

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
    "text.usetex": False,   # ensure external LaTeX is not used
    # and do not select the "pgf" backend
})
from sklearn.metrics import mean_squared_error

# make sure that pdflatex is avaible for matplotlib to use
# import os, shutil
# texbin = "/Library/TeX/texbin"
# if texbin not in os.environ.get("PATH", ""):
#     os.environ["PATH"] = texbin + os.pathsep + os.environ.get("PATH", "")
# print("pdflatex after fix:", shutil.which("pdflatex"))

def mitigate_vals(ideal_vals, measured_vals):
    fit = linregress(measured_vals[[0,10,19]],ideal_vals[[0,10,19]])
    mit_vals = [fit.slope * m + fit.intercept for m in measured_vals]
    mit_vals_err = [np.sqrt((fit.stderr * m)**2 + fit.intercept_stderr**2) for m in measured_vals]
    
    return mit_vals,mit_vals_err

def simulate_counts(circs):
    sim=AerSimulator(method="matrix_product_state")
    return sim.run(circs,shots=1000).result().get_counts()

In [36]:
# import hamiltonians
import matplotlib.pyplot as plt
import numpy as np
# from utils import true_ground_state_energy_and_state

import numpy             as     np
import matplotlib        as     mpl
import matplotlib.pyplot as     plt
from   matplotlib        import rc
from   cycler            import cycler

_widths = {
    # a4paper columnwidth = 426.79135 pt = 5.93 in
    # letterpaper columnwidth = 443.57848 pt = 6.16 in
    'onecolumn': {
        'a4paper' : 5.93,
        'letterpaper' : 6.16
    },
    # a4paper columnwidth = 231.84843 pt = 3.22 in
    # letterpaper columnwidth = 240.24199 pt = 3.34 in
    'twocolumn': {
        'a4paper' : 3.22,
        'letterpaper' : 3.34
    }
}

_wide_widths = {
    # a4paper wide columnwidth = 426.79135 pt = 5.93 in
    # letterpaper wide columnwidth = 443.57848 pt = 6.16 in
    'onecolumn': {
        'a4paper' : 5.93,
        'letterpaper' : 6.16
    },
    # a4paper wide linewidth = 483.69687 pt = 6.72 in
    # letterpaper wide linewidth = 500.48400 pt = 6.95 in
    'twocolumn': {
        'a4paper' : 6.72,
        'letterpaper' : 6.95
    }
}

_fontsizes = {
    10 : {
        'tiny' : 5,
        'scriptsize' : 7,
        'footnotesize' : 8, 
        'small' : 9, 
        'normalsize' : 10,
        'large' : 12, 
        'Large' : 14, 
        'LARGE' : 17,
        'huge' : 20,
        'Huge' : 25
    },
    11 : {
        'tiny' : 6,
        'scriptsize' : 8,
        'footnotesize' : 9, 
        'small' : 10, 
        'normalsize' : 11,
        'large' : 12, 
        'Large' : 14, 
        'LARGE' : 17,
        'huge' :  20,
        'Huge' :  25
    },
    12 : {
        'tiny' : 6,
        'scriptsize' : 8,
        'footnotesize' : 10, 
        'small' : 11, 
        'normalsize' : 12,
        'large' : 14, 
        'Large' : 17, 
        'LARGE' : 20,
        'huge' :  25,
        'Huge' :  25
    }
}

_width         = 1
_wide_width    = 1
_quantumviolet = '#53257F'
_quantumgray   = '#555555'

# Sets up the plot with the fitting arguments so that the font sizes of the plot
# and the font sizes of the document are well aligned
#
#     columns : string = ('onecolumn' | 'twocolumn')
#         the columns you used to set up your quantumarticle, 
#         defaults to 'twocolumn'
#
#     paper : string = ('a4paper' | 'letterpaper')
#         the paper size you used to set up your quantumarticle,
#         defaults to 'a4paper'
#
#     fontsize : int = (10 | 11 | 12)
#         the fontsize you used to set up your quantumarticle as int
#
#     (returns) : dict
#         parameters that can be used for plot adjustments

def global_setup(columns = 'twocolumn', paper = 'a4paper', fontsize = 10):
    plt.rcdefaults()
        
    # Seaborn white is a good base style
    plt.style.use(['seaborn-v0_8-white', './quantum-plots.mplstyle'])
    # plt.style.use(['seaborn-white', './quantum-plots.mplstyle'])
    
    try:        
        # This hackery is necessary so that jupyther shows the plots
        mpl.use("pgf")
        %matplotlib inline
        plt.plot()
        mpl.use("pgf")
    except:
        print('Call to matplotlib.use had no effect')
        
    mpl.interactive(False) 
    
    # Now prepare the styling that depends on the settings of the document
    
    global _width 
    _width = _widths[columns][paper]
    
    global _wide_width 
    _wide_width = _wide_widths[columns][paper]
    
    # Use the default fontsize scaling of LaTeX
    global _fontsizes
    fontsizes = _fontsizes[fontsize]
    
    plt.rcParams['axes.labelsize'] = fontsizes['small']
    plt.rcParams['axes.titlesize'] = fontsizes['large']
    plt.rcParams['xtick.labelsize'] = fontsizes['footnotesize']
    plt.rcParams['ytick.labelsize'] = fontsizes['footnotesize']
    plt.rcParams['font.size'] = fontsizes['small']
    
    return {
            'fontsizes' : fontsizes,
            'colors' : {
                'quantumviolet' : _quantumviolet,
                'quantumgray' : _quantumgray
            }
        }
    

# Sets up the plot with the fitting arguments so that the font sizes of the plot
# and the font sizes of the document are well aligned
#
#     aspect_ratio : float
#         the aspect ratio (width/height) of your plot
#         defaults to the golden ratio
#
#     width_ratio : float in [0, 1]
#         the width of your plot when you insert it into the document, e.g.
#         .8 of the regular width
#         defaults to 1.0
#
#     wide : bool 
#         indicates if the figures spans two columns in twocolumn mode, i.e.
#         when the figure* environment is used, has no effect in onecolumn mode 
#         defaults to False
#
#     (returns) : matplotlib figure object
#         the initialized figure object

def plot_setup(aspect_ratio = 1/1.62, width_ratio = 1.0, wide = False):
    width = (_wide_width if wide else _width) * width_ratio
    height = width * aspect_ratio
           
    return plt.figure(figsize=(width,height), dpi=120, facecolor='white')
    
print('Setup methods loaded')

Setup methods loaded


In [37]:
props = global_setup(columns = 'twocolumn', paper = 'a4paper', fontsize = 10)

print('Global props:')
for key in props:
    print(key, ':')
    for subkey in props[key]:
        print('    ', subkey, ': ', props[key][subkey])

Global props:
fontsizes :
     tiny :  5
     scriptsize :  7
     footnotesize :  8
     small :  9
     normalsize :  10
     large :  12
     Large :  14
     LARGE :  17
     huge :  20
     Huge :  25
colors :
     quantumviolet :  #53257F
     quantumgray :  #555555


In [20]:
def get_circ(x,pars_zero=False):
    circ = QuantumCircuit(7,7)

    circ.h(0)
    circ.h(1)
    circ.h(2)
    circ.h(3)
    circ.h(4)
    circ.h(5)
    circ.h(6)

    circ.cz(0,1)
    circ.cz(0,2)
        
    circ.cz(1,3)
    circ.cz(1,4)
    
    circ.cz(2,5)
    circ.cz(2,6)
    
    circ.sdg(0)
    circ.h(0)
    circ.sdg(0)
    
    circ.sdg(1)
    circ.sdg(2)

    circ.rz(x[0],0)
    circ.h(0)
    circ.measure(0,0)
    with circ.if_test((0,1)):
        circ.x(1)
        circ.z(2)
        circ.z(3)
        circ.z(4)
        
        
    circ.rz(x[1],1)
    circ.h(1)
    circ.measure(1,1)
    with circ.if_test((1,1)):
        circ.x(3)
        
    circ.rz(x[2],2)
    circ.h(2)
    circ.measure(2,2)
    with circ.if_test((2,1)):
        circ.x(6)

    if pars_zero == False:
        circ.ry(x[3],3)
        circ.ry(x[4],4)
        circ.ry(x[5],5)
        circ.ry(x[6],6)
        
        circ.rx(x[7],3)
        circ.rx(x[8],4)
        circ.rx(x[9],5)
        circ.rx(x[10],6)
    
    
    return circ


def get_ham(n_qubits, g, periodic=False):
    from qiskit.quantum_info import SparsePauliOp
    
    op_list = []
    for n in range(n_qubits):
        if n == n_qubits-1:
            if periodic:
                op_list.append(("Y"+"I"*(n_qubits-2)+"Y",(1-g)/2))
                op_list.append(("X"+"I"*(n_qubits-2)+"X",-(1+g)/2))
            else:
                continue
        else:
            op_list.append(("I"*n+"XX"+"I"*(n_qubits-n-2),-(1+g)/2))
            op_list.append(("I"*n+"YY"+"I"*(n_qubits-n-2),(1-g)/2))
    return SparsePauliOp.from_list(op_list)


def get_energy(x,g):
    
    mycirc = get_circ([x,np.pi/2,np.pi/2,0,0,0,0,0,0,0,0])
    hamiltonian = get_ham(4,g,periodic=True)
    mycirc.save_expectation_value(hamiltonian,[3,4,5,6])
    
    return AerSimulator().run(mycirc).result().data()["expectation_value"]

def get_ideal_min(g):
    hamiltonian = get_ham(4,g,periodic=True)
    return np.linalg.eigh(hamiltonian)[0][0]

def get_vqe_min(g):
    from scipy.optimize import minimize
    res = minimize(lambda x: get_energy(x[0],g),[0])
    
    return res.fun


In [21]:
hamiltonian = get_ham(4,1,periodic=True)

In [22]:

def add_counts(exp_counts):
    all_counts = {}
    for counts in exp_counts:
        for c in counts:
            if c in all_counts:
                all_counts[c] += counts[c]
            else:
                all_counts.update({c: counts[c]})
    return all_counts

def filter_counts(exp_counts):
    num_shots = 0
    c_filtered = {}
    for key in exp_counts:
        key_rev = key[::-1]
        if key_rev[0] == "0":
            num_shots += exp_counts[key]
            if key_rev[::-1] in c_filtered:
                c_filtered[key_rev[::-1]] += exp_counts[key]
            else:
                c_filtered.update({key_rev[::-1] : exp_counts[key]})

    for key in c_filtered:
        c_filtered[key] /= num_shots

    return c_filtered

def correct_final_counts(counts, basis = ["z","z","z","z"]):
    cor_counts = {}
            
    for key in counts:
        key_c = key[::-1]
        if basis[0] != "x":
            if key_c[1] == "1":
                if key_c[3] == "0":
                    key_c = key_c[0:3]+"1"+key_c[4:]
                else:
                    key_c = key_c[0:3]+"0"+key_c[4:]
        if basis[-1] != "x":
            if key_c[2] == "1":
                if key_c[6] == "0":
                    key_c = key_c[0:6]+"1"+key_c[7:]
                else:
                    key_c = key_c[0:6]+"0"+key_c[7:]
                
        if key_c[::-1] in cor_counts:
            cor_counts[key_c[::-1]] += counts[key]
        else:
            cor_counts.update({key_c[::-1] : counts[key]})
    
    return cor_counts
    
def get_exp(g,count_list,num_shots=10000):
    
    #XXII
    counts = marginal_counts(count_list[0],[2,3])
    val_xxii=0
    for c in counts:        
        if c.count("1")%2==0:
            val_xxii += counts[c]
        else:
            val_xxii -= counts[c]
    val_xxii /= num_shots

    #YYII
    counts = marginal_counts(count_list[1],[2,3])
    val_yyii=0
    for c in counts:        
        if c.count("1")%2==0:
            val_yyii += counts[c]
        else:
            val_yyii -= counts[c]
    val_yyii /= num_shots

    #IXXI
    counts = marginal_counts(count_list[0],[1,2])
    val_ixxi=0
    for c in counts:        
        if c.count("1")%2==0:
            val_ixxi += counts[c]
        else:
            val_ixxi -= counts[c]
    val_ixxi /= num_shots
    

    #IYYI
    counts = marginal_counts(count_list[1],[1,2])
    val_iyyi=0
    for c in counts:        
        if c.count("1")%2==0:
            val_iyyi += counts[c]
        else:
            val_iyyi -= counts[c]
    val_iyyi /= num_shots

    #IIXX
    counts = marginal_counts(count_list[0],[0,1])
    val_iixx=0
    for c in counts:        
        if c.count("1")%2==0:
            val_iixx += counts[c]
        else:
            val_iixx -= counts[c]
    val_iixx /= num_shots
    

    #IIYY
    counts = marginal_counts(count_list[1],[0,1])
    val_iiyy=0
    for c in counts:        
        if c.count("1")%2==0:
            val_iiyy += counts[c]
        else:
            val_iiyy -= counts[c]
    val_iiyy /= num_shots
    

    #XIIX
    counts = marginal_counts(count_list[0],[0,3])
    val_xiix=0
    for c in counts:        
        if c.count("1")%2==0:
            val_xiix += counts[c]
        else:
            val_xiix -= counts[c]
    val_xiix /= num_shots

    #YIIY
    counts = marginal_counts(count_list[1],[0,3])
    val_yiiy=0
    for c in counts:        
        if c.count("1")%2==0:
            val_yiiy += counts[c]
        else:
            val_yiiy -= counts[c]
    val_yiiy /= num_shots
    
    
    return -(1+g)/2*(val_xxii+val_ixxi+val_iixx+val_xiix)+(1-g)/2*(val_yyii+val_iyyi+val_iiyy+val_yiiy)
    

# SKIP THIS

In [4]:
job_ids_ps_x=['cmrrwxbatjcg008xtxvg',
 'cmrrwyvatjcg008xtxw0',
 'cmrrx0catjcg008xtxwg',
 'cmrrx1mwfj7g008sjk10',
 'cmrrx3479vkg008xgdm0',
 'cmrrx4cnwsvg008dfaq0',
 'cmrrx5mwfj7g008sjk20',
 'cmrrx74nwsvg008dfar0',
 'cmrrx8nnwsvg008dfarg',
 'cmrrxgpr54bg008mjfcg',
 'cmrrxj6atjcg008xtxzg',
 'cmrrxkpnwsvg008dfav0',
 'cmrrxmpr54bg008mjfe0',
 'cmrrxp6b37pg008q8z7g',
 'cmrrxqpr54bg008mjfeg',
 'cmrrxrznwsvg008dfaw0',
 'cmrrxtfb37pg008q8z80',
 'cmrrxvqnwsvg008dfax0',
 'cmrrxwzb37pg008q8z8g',
 'cmrrxy779vkg008xgdp0']

job_ids_ps_y=['cmrrwy379vkg008xgdkg',
 'cmrrwzkr54bg008mjfb0',
 'cmrrx0watjcg008xtxx0',
 'cmrrx2catjcg008xtxxg',
 'cmrrx3mnwsvg008dfapg',
 'cmrrx4wb37pg008q8z6g',
 'cmrrx6mnwsvg008dfaqg',
 'cmrrx7wr54bg008mjfbg',
 'cmrrx9dnwsvg008dfas0',
 'cmrrxheb37pg008q8z70',
 'cmrrxjpnwsvg008dfat0',
 'cmrrxm6r54bg008mjfdg',
 'cmrrxnp79vkg008xgdn0',
 'cmrrxpywfj7g008sjk30',
 'cmrrxrfwfj7g008sjk3g',
 'cmrrxsznwsvg008dfawg',
 'cmrrxtz79vkg008xgdng',
 'cmrrxw7wfj7g008sjk40',
 'cmrrxxqatjcg008xty1g',
 'cmrry00b37pg008q8z90']

job_ids_dynamic_ps_x=['cmrrw1g79vkg008xgdgg',
 'cmrrw2rwfj7g008sjjw0',
 'cmrrw40b37pg008q8z30',
 'cmrrw6rb37pg008q8z3g',
 'cmrrw89wfj7g008sjjwg',
 'cmrrw9hnwsvg008dfak0',
 'cmrrwb1wfj7g008sjjx0',
 'cmrrwc179vkg008xgdhg',
 'cmrrwdhatjcg008xtxr0',
 'cmrrweswfj7g008sjjy0',
 'cmrrwgjwfj7g008sjjyg',
 'cmrrwhjnwsvg008dfamg',
 'cmrrwka79vkg008xgdj0',
 'cmrrwmjatjcg008xtxsg',
 'cmrrwnjwfj7g008sjk00',
 'cmrrwq2b37pg008q8z50',
 'cmrrwr3atjcg008xtxt0',
 'cmrrwsbr54bg008mjfa0',
 'cmrrwtv79vkg008xgdk0',
 'cmrrww3b37pg008q8z5g']

job_ids_dynamic_ps_y=['cmrrw20b37pg008q8z2g',
 'cmrrw3gr54bg008mjf70',
 'cmrrw58nwsvg008dfaj0',
 'cmrrw7gr54bg008mjf80',
 'cmrrw8snwsvg008dfajg',
 'cmrrwa9r54bg008mjf8g',
 'cmrrwbhwfj7g008sjjxg',
 'cmrrwcsr54bg008mjf90',
 'cmrrwe9nwsvg008dfakg',
 'cmrrwfhb37pg008q8z40',
 'cmrrwh2nwsvg008dfam0',
 'cmrrwjjb37pg008q8z4g',
 'cmrrwktatjcg008xtxs0',
 'cmrrwn2wfj7g008sjjzg',
 'cmrrwpjnwsvg008dfan0',
 'cmrrwqjnwsvg008dfang',
 'cmrrwrvatjcg008xtxtg',
 'cmrrwtbwfj7g008sjk0g',
 'cmrrwvknwsvg008dfap0',
 'cmrrwwkb37pg008q8z60']

job_ids_dynamic_x=['cmrrv54wfj7g008sjjmg',
 'cmrrv6watjcg008xtxm0',
 'cmrrv8nwfj7g008sjjn0',
 'cmrrva5r54bg008mjf20',
 'cmrrvbnb37pg008q8yyg',
 'cmrrvddatjcg008xtxn0',
 'cmrrvenatjcg008xtxng',
 'cmrrvfxb37pg008q8z00',
 'cmrrvhe79vkg008xgdf0',
 'cmrrvjpb37pg008q8z0g',
 'cmrrvkywfj7g008sjjqg',
 'cmrrvn6r54bg008mjf3g',
 'cmrrvpewfj7g008sjjrg',
 'cmrrvr7r54bg008mjf5g',
 'cmrrvsfnwsvg008dfah0',
 'cmrrvtqr54bg008mjf60',
 'cmrrvw7atjcg008xtxp0',
 'cmrrvxfatjcg008xtxpg',
 'cmrrvyqwfj7g008sjjv0',
 'cmrrw0079vkg008xgdg0']

job_ids_dynamic_y=['cmrrv64nwsvg008dfaf0',
 'cmrrv7matjcg008xtxmg',
 'cmrrv9dnwsvg008dfafg',
 'cmrrvax79vkg008xgde0',
 'cmrrvcnb37pg008q8yzg',
 'cmrrvdxwfj7g008sjjng',
 'cmrrvf5r54bg008mjf30',
 'cmrrvgp79vkg008xgdeg',
 'cmrrvhywfj7g008sjjpg',
 'cmrrvk6wfj7g008sjjq0',
 'cmrrvme79vkg008xgdfg',
 'cmrrvnpnwsvg008dfag0',
 'cmrrvq6b37pg008q8z10',
 'cmrrvrzwfj7g008sjjs0',
 'cmrrvszb37pg008q8z1g',
 'cmrrvvfwfj7g008sjjsg',
 'cmrrvwzwfj7g008sjjtg',
 'cmrrvy7nwsvg008dfahg',
 'cmrrvzfwfj7g008sjjvg',
 'cmrrw0rb37pg008q8z20']


In [5]:
from qiskit import qasm3

In [6]:
circs_ps_x = [[qasm3.dumps(c) for c in provider.retrieve_job(job_id).circuits()] for job_id in job_ids_ps_x]
circs_ps_y = [[qasm3.dumps(c) for c in provider.retrieve_job(job_id).circuits()] for job_id in job_ids_ps_y]

circs_dynamic_x = [[qasm3.dumps(c) for c in provider.retrieve_job(job_id).circuits()] for job_id in job_ids_dynamic_x]
circs_dynamic_y = [[qasm3.dumps(c) for c in provider.retrieve_job(job_id).circuits()] for job_id in job_ids_dynamic_y]

circs_dynamic_ps_x = [[qasm3.dumps(c) for c in provider.retrieve_job(job_id).circuits()] for job_id in job_ids_dynamic_ps_x]
circs_dynamic_ps_y = [[qasm3.dumps(c) for c in provider.retrieve_job(job_id).circuits()] for job_id in job_ids_dynamic_ps_y]

NameError: name 'job_ids_ps_x' is not defined

In [10]:
counts_ps_x = [provider.retrieve_job(job_id).result().get_counts() for job_id in job_ids_ps_x]
counts_ps_y = [provider.retrieve_job(job_id).result().get_counts() for job_id in job_ids_ps_y]

counts_dynamic_x = [provider.retrieve_job(job_id).result().get_counts() for job_id in job_ids_dynamic_x]
counts_dynamic_y = [provider.retrieve_job(job_id).result().get_counts() for job_id in job_ids_dynamic_y]

counts_dynamic_ps_x = [provider.retrieve_job(job_id).result().get_counts() for job_id in job_ids_dynamic_ps_x]
counts_dynamic_ps_y = [provider.retrieve_job(job_id).result().get_counts() for job_id in job_ids_dynamic_ps_y]

NameError: name 'job_ids_ps_x' is not defined

In [28]:
res_ps_x = {"circs": circs_ps_x,"counts": counts_ps_x}
res_ps_y = {"circs": circs_ps_y,"counts": counts_ps_y}

res_dynamic_x = {"circs": circs_dynamic_x,"counts": counts_dynamic_x}
res_dynamic_y = {"circs": circs_dynamic_y,"counts": counts_dynamic_y}

res_dynamic_ps_x = {"circs": circs_dynamic_ps_x,"counts": counts_dynamic_ps_x}
res_dynamic_ps_y = {"circs": circs_dynamic_ps_y,"counts": counts_dynamic_ps_y}


In [11]:
import pickle

with open('xy_dynamic_y.pickle', 'wb') as handle:   
    pickle.dump(res_dynamic_y, handle)

NameError: name 'res_dynamic_y' is not defined

# CONTINUE HERE

In [23]:
import pickle

with open('xy_ps_x.pickle', 'rb') as file:   
    res_ps_x=pickle.load(file)
with open('xy_ps_y.pickle', 'rb') as file:   
    res_ps_y=pickle.load(file)
    
with open('xy_dynamic_x.pickle', 'rb') as file:   
    res_dynamic_x=pickle.load(file)
with open('xy_dynamic_y.pickle', 'rb') as file:   
    res_dynamic_y=pickle.load(file)    
    
with open('xy_dynamic_ps_x.pickle', 'rb') as file:   
    res_dynamic_ps_x=pickle.load(file)
with open('xy_dynamic_ps_y.pickle', 'rb') as file:   
    res_dynamic_ps_y=pickle.load(file)    


In [24]:
from qiskit import qasm3
from tqdm import tqdm
        
circs_ps_x = [[qasm3.loads(c) for c in circ] for circ in tqdm(res_ps_x["circs"])]
circs_ps_y = [[qasm3.loads(c) for c in circ] for circ in tqdm(res_ps_y["circs"])]

counts_ps_x = res_ps_x["counts"]
counts_ps_y = res_ps_y["counts"]

circs_dynamic_x = [[qasm3.loads(c) for c in circ] for circ in tqdm(res_dynamic_x["circs"])]
circs_dynamic_y = [[qasm3.loads(c) for c in circ] for circ in tqdm(res_dynamic_y["circs"])]

counts_dynamic_x = res_dynamic_x["counts"]
counts_dynamic_y = res_dynamic_y["counts"]

circs_dynamic_ps_x = [[qasm3.loads(c) for c in circ] for circ in tqdm(res_dynamic_ps_x["circs"])]
circs_dynamic_ps_y = [[qasm3.loads(c) for c in circ] for circ in tqdm(res_dynamic_ps_y["circs"])]

counts_dynamic_ps_x = res_dynamic_ps_x["counts"]
counts_dynamic_ps_y = res_dynamic_ps_y["counts"]

100%|██████████| 20/20 [00:31<00:00,  1.59s/it]


In [25]:
points=list(np.linspace(0,np.pi/2,11))+list(np.linspace(np.pi/2,np.pi,10))[1:]

# Plot theory curve true minimum vs vqe minimum (simulated)

# Y Hamiltonian (g = -1.0)

In [26]:
measured_val_dynamic=[]

for n in range(20):
    counts_x = marginal_counts(correct_final_counts(add_counts(counts_dynamic_x[n]),["x"]*4),[3,4,5,6])
    counts_y = marginal_counts(correct_final_counts(add_counts(counts_dynamic_y[n]),["y"]*4),[3,4,5,6])
    measured_val_dynamic.append(get_exp(-1.0,[counts_x,counts_y],num_shots=100*1000))
    
# #just to test for consistency
# ideal_val_dynamic=[]
# from tqdm import tqdm
# for n in tqdm(range(20)):
    
#     counts_x = marginal_counts(correct_final_counts(add_counts(simulate_counts(circs_dynamic_x[n])),["x"]*4),[3,4,5,6])
#     counts_y = marginal_counts(correct_final_counts(add_counts(simulate_counts(circs_dynamic_y[n])),["y"]*4),[3,4,5,6])
#     ideal_val_dynamic.append(get_exp(0.2,[counts_x,counts_y],num_shots=100*1000))

measured_val_ps=[]

for n in range(20):
    counts_x = marginal_counts(filter_counts(correct_final_counts(
        add_counts(counts_ps_x[n]),["x"]*4)),[3,4,5,6])
    counts_y = marginal_counts(filter_counts(correct_final_counts(
        add_counts(counts_ps_y[n]),["y"]*4)),[3,4,5,6])
    measured_val_ps.append(get_exp(-1.0,[counts_x,counts_y],num_shots=1))
    
# #just to test for consistency
# ideal_val_ps=[]
# from tqdm import tqdm
# for n in tqdm(range(20)):
    
#     counts_x = marginal_counts(filter_counts(correct_final_counts(
#         add_counts(simulate_counts(circs_ps_x[n])),["x"]*4)),[3,4,5,6])
#     counts_y = marginal_counts(filter_counts(correct_final_counts(
#         add_counts(simulate_counts(circs_ps_y[n])),["y"]*4)),[3,4,5,6])
#     ideal_val_ps.append(get_exp(0.2,[counts_x,counts_y],num_shots=1))

ideal_val_y = np.array([get_energy(x,-1.0) for x in points])
measured_val_ps_y = np.array(measured_val_ps)
measured_val_dynamic_y = np.array(measured_val_dynamic)

mit_vals_y = mitigate_vals(ideal_val_y,measured_val_ps_y)
mit_vals_y_dyn = mitigate_vals(ideal_val_y,measured_val_dynamic_y)


# X Hamiltonian (g=1.0)

In [27]:
measured_val_dynamic=[]

for n in range(20):
    counts_x = marginal_counts(correct_final_counts(add_counts(counts_dynamic_x[n]),["x"]*4),[3,4,5,6])
    counts_y = marginal_counts(correct_final_counts(add_counts(counts_dynamic_y[n]),["y"]*4),[3,4,5,6])
    measured_val_dynamic.append(get_exp(1.0,[counts_x,counts_y],num_shots=100*1000))
    
# #just to test for consistency
# ideal_val_dynamic=[]
# from tqdm import tqdm
# for n in tqdm(range(20)):
    
#     counts_x = marginal_counts(correct_final_counts(add_counts(simulate_counts(circs_dynamic_x[n])),["x"]*4),[3,4,5,6])
#     counts_y = marginal_counts(correct_final_counts(add_counts(simulate_counts(circs_dynamic_y[n])),["y"]*4),[3,4,5,6])
#     ideal_val_dynamic.append(get_exp(0.2,[counts_x,counts_y],num_shots=100*1000))

measured_val_ps=[]

for n in range(20):
    counts_x = marginal_counts(filter_counts(correct_final_counts(
        add_counts(counts_ps_x[n]),["x"]*4)),[3,4,5,6])
    counts_y = marginal_counts(filter_counts(correct_final_counts(
        add_counts(counts_ps_y[n]),["y"]*4)),[3,4,5,6])
    measured_val_ps.append(get_exp(1.0,[counts_x,counts_y],num_shots=1))
    
# #just to test for consistency
# ideal_val_ps=[]
# from tqdm import tqdm
# for n in tqdm(range(20)):
    
#     counts_x = marginal_counts(filter_counts(correct_final_counts(
#         add_counts(simulate_counts(circs_ps_x[n])),["x"]*4)),[3,4,5,6])
#     counts_y = marginal_counts(filter_counts(correct_final_counts(
#         add_counts(simulate_counts(circs_ps_y[n])),["y"]*4)),[3,4,5,6])
#     ideal_val_ps.append(get_exp(0.2,[counts_x,counts_y],num_shots=1))

ideal_val_x = np.array([get_energy(x,1.0) for x in points])
measured_val_ps_x = np.array(measured_val_ps)
measured_val_dynamic_x = np.array(measured_val_dynamic)

mit_vals_x = mitigate_vals(ideal_val_x,measured_val_ps_x)
mit_vals_x_dyn = mitigate_vals(ideal_val_x,measured_val_dynamic_x)

In [28]:
# ax0 dyn y
fit = linregress(measured_val_dynamic_y[[0,10,19]],ideal_val_y[[0,10,19]])
x = measured_val_dynamic_y
ideal_y = fit.intercept + fit.slope*x
my_y = ideal_val_y

print('ax0: ', mean_squared_error(my_y, ideal_y))

# ax1 ps y
fit = linregress(measured_val_ps_y[[0,10,19]],ideal_val_y[[0,10,19]])
x = measured_val_ps_y
ideal_y = fit.intercept + fit.slope*x
my_y = ideal_val_y

print('ax1: ', mean_squared_error(my_y, ideal_y))

# ax2 dyn x
fit = linregress(measured_val_dynamic_x[[0,10,19]],ideal_val_x[[0,10,19]])
x = measured_val_dynamic_x
ideal_x = fit.intercept + fit.slope*x
my_x = ideal_val_x

print('ax2: ', mean_squared_error(my_x, ideal_x))

# ax3 ps x
fit = linregress(measured_val_ps_x[[0,10,19]],ideal_val_x[[0,10,19]])
x = measured_val_ps_x
ideal_x = fit.intercept + fit.slope*x
my_x = ideal_val_x

print('ax3: ', mean_squared_error(my_x, ideal_x))

ax0:  0.052294021056088456
ax1:  0.0005134995689535358
ax2:  0.007965197058207021
ax3:  0.003221670134025237


In [29]:
# get the colors of the matplotlib standard palette
prop_cycle = plt.rcParams['axes.prop_cycle']

blue = prop_cycle.by_key()['color'][0]
orange = prop_cycle.by_key()['color'][1]
green = prop_cycle.by_key()['color'][2]


In [ ]:
fig = plot_setup(width_ratio=1.2)


ax0 = fig.add_subplot(2,2,1)
ax1 = fig.add_subplot(2,2,2, sharey=ax0)

ax3 = fig.add_subplot(2,2,3)
ax4 = fig.add_subplot(2,2,4, sharey=ax3)

fit = linregress(measured_val_dynamic_y[[0,10,19]],ideal_val_y[[0,10,19]])
x = np.linspace(measured_val_dynamic_y.min(),measured_val_dynamic_y.max())
ax0.plot(x, fit.intercept + fit.slope*x, '--', color=_quantumgray,zorder=1)
ax0.scatter(measured_val_dynamic_y,ideal_val_y, label="data",marker="o",zorder=2)
ax0.scatter(measured_val_dynamic_y[[0,10,19]],ideal_val_y[[0,10,19]], label="Clifford", marker="x",zorder=3)
ax0.set_ylabel(r"$\langle H_Y \rangle_{ideal}$")
ax0.set_xlabel(r"$\langle H_Y \rangle_{meas}$")

fit = linregress(measured_val_ps_y[[0,10,19]],ideal_val_y[[0,10,19]])
x = np.linspace(measured_val_ps_y.min(),measured_val_ps_y.max())
ax1.plot(x, fit.intercept + fit.slope*x, '--', color=_quantumgray, zorder=1)
ax1.scatter(measured_val_ps_y,ideal_val_y, label="data",marker="o", zorder=2)
ax1.scatter(measured_val_ps_y[[0,10,19]],ideal_val_y[[0,10,19]], label="Clifford", marker="x", zorder=3)
ax1.tick_params(labelleft=False)
ax1.set_xlabel(r"$\langle H_Y \rangle_{meas}$")


ax1.legend(loc="lower right",labelspacing=0.05,borderpad=0.1,handletextpad=0.02,)

fit = linregress(measured_val_dynamic_x[[0,10,19]],ideal_val_x[[0,10,19]])
x = np.linspace(measured_val_dynamic_x.min(),measured_val_dynamic_x.max())
ax3.plot(x, fit.intercept + fit.slope*x, '--', color=_quantumgray, zorder=1)
ax3.scatter( measured_val_dynamic_x,ideal_val_x,label="data point",marker="o", zorder=2)
ax3.scatter(measured_val_dynamic_x[[0,10,19]],ideal_val_x[[0,10,19]], label="Clifford point", marker="x", zorder=3)
ax3.set_ylabel(r"$\langle H_X \rangle_{ideal}$")
ax3.set_xlabel(r"$\langle H_X \rangle_{meas}$")

fit = linregress(measured_val_ps_x[[0,10,19]],ideal_val_x[[0,10,19]])
x = np.linspace(measured_val_ps_x.min(),measured_val_ps_x.max())
ax4.plot(x, fit.intercept + fit.slope*x, '--', color=_quantumgray, zorder=1)
ax4.scatter(measured_val_ps_x,ideal_val_x, label="data point",marker="o", zorder=2)
ax4.scatter(measured_val_ps_x[[0,10,19]],ideal_val_x[[0,10,19]], label="Clifford point", marker="x", zorder=3)
ax4.set_xlabel(r"$\langle H_X \rangle_{meas}$")

ax4.tick_params(labelleft=False)

fig.subplots_adjust(wspace=0.1, hspace=0.5)
y = np.sqrt(6.72 * 0.5) / 100 * 2.2

fig.savefig("xy_hardware_double_col_corr.pdf",bbox_inches='tight')

In [ ]:
fig = plot_setup(aspect_ratio=1.5,width_ratio=1)

ax2 = fig.add_subplot(2,1,1)
ax5 = fig.add_subplot(2,1,2)

ax2.scatter(points,measured_val_dynamic_y,marker=".",label="dynamic")
ax2.scatter(points,measured_val_ps_y,marker=".",label="ps")
ax2.plot(points,[get_energy(x,-1.0) for x in points], color=_quantumgray,linestyle="--", dashes=(5, 5))
ax2.errorbar(x=points, y=mit_vals_y_dyn[0], yerr = mit_vals_y_dyn[1],fmt=",",label="mitigated",capsize=3,color=blue)
ax2.errorbar(x=np.asarray(points)+0.005, y=mit_vals_y[0], yerr = mit_vals_y[1],fmt=",",label="mitigated",capsize=3,color=orange)
ax2.set_ylabel(r"$\langle H_Y \rangle_{meas}$")


ax5.plot(points,[get_energy(x,1.0) for x in points],color=_quantumgray,linestyle="--", label='ideal', dashes=(5, 5))
ax5.scatter(points,measured_val_dynamic_x,marker=".",label="dynamic")
ax5.scatter(points,measured_val_ps_x,marker=".",label="ps")
ax5.errorbar(x=points, y=mit_vals_x_dyn[0], yerr = mit_vals_x_dyn[1],fmt=",",capsize=3,label="mit. dyn.",color=blue)
ax5.errorbar(x=np.asarray(points)+0.005, y=mit_vals_x[0], yerr = mit_vals_x[1],fmt=",",capsize=3,label="mit. ps",color=orange)

ax5.legend(loc="lower right")
ax5.set_xlabel(r"$\theta$")
ax5.set_ylabel(r"$\langle H_X \rangle_{meas}$")

fig.subplots_adjust(hspace=0.2)

fig.savefig("xy_fig_both.pdf",bbox_inches='tight')


In [39]:
gs = np.linspace(0,1,10)

ham_exp_ps = [list(0.5*(1+g)*np.asarray(mit_vals_x[0]) + 0.5*(1-g)*np.asarray(mit_vals_y[0])) for g in gs]
ham_min_ps = [min(h) for h in ham_exp_ps]
ham_min_ind = [h.index(min(h)) for h in ham_exp_ps]
ham_min_err_ps = [np.sqrt((0.5*(1+gs[j])*mit_vals_x[1][i])**2 + (0.5*(1-gs[j])*mit_vals_y[1][i])**2) for j,i in enumerate(ham_min_ind)]

In [40]:
ham_exp_dyn = [list(0.5*(1+g)*np.asarray(mit_vals_x_dyn[0]) + 0.5*(1-g)*np.asarray(mit_vals_y_dyn[0])) for g in gs]
ham_min_dyn = [min(h) for h in ham_exp_dyn]
ham_min_ind = [h.index(min(h)) for h in ham_exp_dyn]
ham_min_err_dyn = [np.sqrt((0.5*(1+gs[j])*mit_vals_x_dyn[1][i])**2 + (0.5*(1-gs[j])*mit_vals_y_dyn[1][i])**2) for j,i in enumerate(ham_min_ind)]

In [41]:
fig = plot_setup(width_ratio=1.2)
ax = fig.add_subplot(1,1,1)

ax.plot(np.linspace(0,1),[get_ideal_min(x) for x in np.linspace(0,1)],label="true $E_{gs}$",color=_quantumgray,linestyle="dashed",zorder=1)

plt.errorbar(x=np.linspace(0,1,10)-0.005, y=ham_min_dyn, yerr = ham_min_err_dyn, fmt=",",label="dynamic",capsize=3)
plt.errorbar(x=np.linspace(0,1,10)+0.005, y=ham_min_ps, yerr = ham_min_err_ps, fmt=",",label="ps",capsize=3)

ax.scatter(np.linspace(0,1,10),[get_vqe_min(x) for x in np.linspace(0,1,10)],label="ideal", color=prop_cycle.by_key()['color'][2],zorder=4)

ax.set_xlabel(r"$g$")
ax.set_ylabel(r"$\langle H_{XY} \rangle$")
ax.legend()
fig.savefig("xy_ideal_vqe_hwdata.pdf",bbox_inches='tight')